#### Image Read
#### resize
#### crop
#### rectangle draw
#### webcam capture
#### webcam feed

In [56]:
import cv2

##### READ

In [57]:
img = cv2.imread("/Users/par_04/Pictures/pic.png")

In [58]:
if img is None:
    raise RuntimeError("Image not found")

In [59]:
img

array([[[254, 254, 254],
        [254, 254, 254],
        [255, 255, 255],
        ...,
        [223, 210, 186],
        [223, 210, 186],
        [223, 210, 186]],

       [[253, 253, 253],
        [254, 254, 254],
        [254, 254, 254],
        ...,
        [223, 210, 186],
        [223, 210, 186],
        [223, 210, 186]],

       [[253, 253, 253],
        [253, 253, 253],
        [254, 254, 254],
        ...,
        [224, 211, 187],
        [224, 211, 187],
        [223, 210, 186]],

       ...,

       [[158, 201, 218],
        [157, 200, 217],
        [159, 199, 217],
        ...,
        [159, 149, 126],
        [159, 149, 126],
        [159, 149, 126]],

       [[157, 200, 217],
        [157, 200, 217],
        [158, 198, 216],
        ...,
        [158, 148, 125],
        [158, 148, 125],
        [158, 148, 125]],

       [[156, 199, 216],
        [156, 199, 216],
        [158, 198, 216],
        ...,
        [158, 148, 125],
        [158, 148, 125],
        [158, 148, 125]]

In [60]:
type(img)

numpy.ndarray

In [61]:
img.shape

(1076, 1614, 3)

#### RESIZE

cv2.INTER_AREA <br>is an OpenCV interpolation method that uses pixel area relation for resampling. It is the gold standard for downscaling/shrinking images because it averages pixel values over a region, preventing jagged edges and removing moiré patterns

In [62]:
resized= cv2.resize(img, (224,224), interpolation=cv2.INTER_AREA)

In [63]:
resized.shape

(224, 224, 3)

##### saving a greyscale image

In [64]:
grey_img=cv2.cvtColor(resized,cv2.COLOR_RGB2GRAY)
cv2.imwrite("resized.png", resized)

True

In [65]:

import cv2
print("Before destroy")

cv2.destroyAllWindows()

print("After destroy")

Before destroy
After destroy


In [66]:

import cv2
import numpy as np

img = np.zeros((300, 300, 3), dtype=np.uint8)

while True:
    cv2.imshow("test", img)

    key = cv2.waitKey(20)

    if key != -1:
        print("Key =", key)

    if key == 27:
        print("ESC detected")
        break

print("Exited loop")

Key = 27
ESC detected
Exited loop


In [67]:

import cv2
import numpy as np

img = np.zeros((500,500,3), dtype=np.uint8)

while True:
    cv2.imshow("test", img)

    key = cv2.waitKey(20)

    if key == 27:
        print("ESC")
        break

cv2.destroyAllWindows()

ESC


##### WEBCAM CAPTURE->GREY-> TAKE IMAGE after 10 secs ->CROP -> SAVE


In [68]:
import time

cap=cv2.VideoCapture(0) # capture video from the main cam
if not cap.isOpened():
    raise RuntimeError("Cam not accessible")
start_time= time.time()
captured_frame= None

# cam loop:

while True:
    ret, frame= cap.read()
    frame= cv2.cvtColor(frame,cv2.COLOR_BGR2GRAY)
    if not ret:
        break
    elapsed= time.time()-start_time
    remaining=int(5-elapsed)
    if remaining >=0:
        # text putting
        cv2.putText(frame, f"capturing in {remaining}", (50,80), cv2.FONT_HERSHEY_SCRIPT_COMPLEX,3,(255,255,255),3)
        cv2.imshow("cam feed", frame) #image window show
    else:
        captured_frame=frame.copy()
        break
    if cv2.waitKey(1) & 0xFF == 27:
        # 0xFF is 11111111 
         #  This function tells OpenCV to wait exactly 1 milliseconds for an esc
        break
cap.release()
cv2.destroyAllWindows()
cv2.imwrite("captured.png",captured_frame)
img=cv2.imread("captured.png")
clone=img.copy() # permanent backup

draw=False
x0,y0=-1,-1

# clone is the IMAGE
def rect(event, x,y, flags, params):
    global x0, y0, img, draw
    # Whenever I use drawing, x0, y0, or img in this function,
    #  use the variables defined outside the function.
    if event==cv2.EVENT_LBUTTONDOWN:
        draw =True
        x0,y0=x,y   # current x,y

    elif event==cv2.EVENT_MOUSEMOVE and draw:
        img=clone.copy()    # img used for live rendering so everytime a fresh image is copied
        side_x=abs(x-x0)
        side_y=abs(y-y0)
        x1=x0+side_x if x>=x0 else x0-side_x
        y1=y0+side_y if y>=y0 else y0-side_y
        # live rect renderring

        cv2.rectangle(img,(x0,y0),(x1,y1), (0,0,255), 2)
    elif event == cv2.EVENT_LBUTTONUP:
        draw=False
        side_x=abs(x-x0)
        side_y=abs(y-y0)
        x1=x0+side_x if x>=x0 else x0-side_x
        y1=y0+side_y if y>=y0 else y0-side_y
        #vertices
        pts=[(x0,y0),(x1,y0),(x1,y1),(x0,y1)]
        print("Clockwise coords:")
        for p in pts:
            print(p)
            
        #cropping done
        img=img[y0:y1,x0:x1]


#  window NAme mattters
window_name="Rect"
cv2.namedWindow(window_name)
# whenever this event happens call the function rect
cv2.setMouseCallback(window_name, rect)

# display the rendered rect untill i close
while True:
    cv2.imshow(window_name, img)
    if cv2.waitKey(1) & 0xFF ==27:  # checking upto 8 bits
        break
# cv2.destroyAllWindows()



Clockwise coords:
(486, 291)
(1459, 291)
(1459, 756)
(486, 756)


##### WEBCAM CAPTURE-> TAKE IMAGE-> DRAW RECTANGLE

In [69]:
import time

cap=cv2.VideoCapture(0) # capture video from the main cam
if not cap.isOpened():
    raise RuntimeError("Cam not accessible")
start_time= time.time()
captured_frame= None

# cam loop:

while True:
    ret, frame= cap.read()
    frame= cv2.cvtColor(frame,cv2.COLOR_BGR2GRAY)
    if not ret:
        break
    elapsed= time.time()-start_time
    remaining=int(5-elapsed)
    if remaining >=0:
        # text putting
        cv2.putText(frame, f"capturing in {remaining}", (50,80), cv2.FONT_HERSHEY_SCRIPT_COMPLEX,3,(255,255,255),3)
        cv2.imshow("cam feed", frame) #image window show
    else:
        captured_frame=frame.copy()
        break
    if cv2.waitKey(1) & 0xFF == 27:
        # 0xFF is 11111111 
         #  This function tells OpenCV to wait exactly 1 milliseconds for an esc
        break
cap.release()
cv2.destroyAllWindows()
cv2.imwrite("captured.png",captured_frame)
img=cv2.imread("captured.png")
clone=img.copy() # permanent backup

draw=False
x0,y0=-1,-1

# clone is the IMAGE
def rect(event, x,y, flags, params):
    global x0, y0, img, draw
    # Whenever I use drawing, x0, y0, or img in this function,
    #  use the variables defined outside the function.
    if event==cv2.EVENT_LBUTTONDOWN:
        draw =True
        x0,y0=x,y   # current x,y

    elif event==cv2.EVENT_MOUSEMOVE and draw:
        img=clone.copy()    # img used for live rendering so everytime a fresh image is copied
        side_x=abs(x-x0)
        side_y=abs(y-y0)
        x1=x0+side_x if x>=x0 else x0-side_x
        y1=y0+side_y if y>=y0 else y0-side_y
        # live rect renderring

        cv2.rectangle(img,(x0,y0),(x1,y1), (0,0,255), 2)
    elif event == cv2.EVENT_LBUTTONUP:
        draw=False
        side_x=abs(x-x0)
        side_y=abs(y-y0)
        x1=x0+side_x if x>=x0 else x0-side_x
        y1=y0+side_y if y>=y0 else y0-side_y
        #vertices
        pts=[(x0,y0),(x1,y0),(x1,y1),(x0,y1)]
        print("Clockwise coords:")
        for p in pts:
            print(p)
            
        # render the rects on the image
        cv2.rectangle(img,(x0,y0),(x1,y1), (255,0,255), 2)


#  window NAme mattters
window_name="Rect"
cv2.namedWindow(window_name)
# whenever this event happens call the function rect
cv2.setMouseCallback(window_name, rect)

# display the rendered rect untill i close
while True:
    cv2.imshow(window_name, img)
    if cv2.waitKey(1) & 0xFF ==27:  # checking upto 8 bits
        break
# cv2.destroyAllWindows()



Clockwise coords:
(485, 384)
(1585, 384)
(1585, 742)
(485, 742)


KeyboardInterrupt: 